In [ ]:
!pip install timm -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import timm
import torch
import torch.nn as nn
import numpy as np
import os
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torch.optim import AdamW
from tqdm.notebook import tqdm


In [ ]:
# Define the path to your data in Google Drive
drive_path = "/content/drive/My Drive/LULC_segmentation"
train_images_path = os.path.join(drive_path, "train_images.npy")
train_masks_path = os.path.join(drive_path, "train_masks.npy")
test_images_path = os.path.join(drive_path, "test_images.npy")
test_masks_path = os.path.join(drive_path, "test_masks.npy")
val_images_path = os.path.join(drive_path, "valid_images.npy")
val_masks_path = os.path.join(drive_path, "valid_masks.npy")
# Load all image and mask arrays
train_images = np.load(train_images_path)
train_masks = np.load(train_masks_path)

val_images = np.load(val_images_path)
val_masks = np.load(val_masks_path)

test_images = np.load(test_images_path)
test_masks = np.load(test_masks_path)

# Define the class labels and their mapping
labels = {
    0: "crops",
    1: "bare",
    2: "built",
    3: "water"
}
num_labels = len(labels)

In [ ]:
from torchvision import transforms

class LULCSegmentationDataset(Dataset):
    def __init__(self, images, masks):
        self.images = images
        self.masks = masks

        # Define normalization transform manually
        self.transform = transforms.Compose([
            transforms.ToTensor(),  # Converts (H, W, C) in [0,255] to (C, H, W) in [0,1]
            transforms.Normalize(mean=[0.485, 0.456, 0.406],  # ImageNet mean
                                 std=[0.229, 0.224, 0.225])   # ImageNet std
        ])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx]
        mask = self.masks[idx]

        # Normalize the image
        image = self.transform(image)

        # Convert mask to tensor
        mask = torch.tensor(mask, dtype=torch.long)

        return {'pixel_values': image, 'labels': mask}


In [ ]:

train_dataset = LULCSegmentationDataset(train_images, train_masks)
val_dataset   = LULCSegmentationDataset(val_images, val_masks)
test_dataset  = LULCSegmentationDataset(test_images, test_masks)

train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_dataloader   = DataLoader(val_dataset, batch_size=16, shuffle=False)
test_dataloader  = DataLoader(test_dataset, batch_size=1, shuffle=False)

In [ ]:
import timm
import torch.nn as nn
import torch

class MobileViTSegmentationModel(nn.Module):
    def __init__(self, num_classes):
        super(MobileViTSegmentationModel, self).__init__()

        # Load MobileViT backbone
        self.backbone = timm.create_model('mobilevit_xxs', pretrained=True, features_only=True)

        # Get output channels from last stage
        backbone_out_channels = self.backbone.feature_info[-1]['num_chs']

        # Simple segmentation head
        self.segmentation_head = nn.Sequential(
            nn.Conv2d(backbone_out_channels, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, num_classes, kernel_size=1)
        )

    def forward(self, x):
        features = self.backbone(x)[-1]
        out = self.segmentation_head(features)

        # Upsample to input size
        out = nn.functional.interpolate(out, size=x.shape[2:], mode="bilinear", align_corners=False)
        return out


In [ ]:
# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MobileViTSegmentationModel(num_classes=num_labels)
model = model.to(device)


In [ ]:
from torch.nn import functional as F
import copy

optimizer = AdamW(model.parameters(), lr=0.00006)

patience = 8
best_val_loss = float("inf")
epochs_no_improve = 0
num_epochs = 200

train_losses = []
val_losses = []
loss_fn = torch.nn.CrossEntropyLoss()

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch + 1}/{num_epochs}")
    train_loss = 0
    val_loss = 0

    # Training
    model.train()
    for batch in tqdm(train_dataloader):
        pixel_values = batch["pixel_values"].to(device)
        labels = batch["labels"].to(device)

        logits = model(pixel_values)
        loss = loss_fn(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    avg_train_loss = train_loss / len(train_dataloader)

    # Validation
    model.eval()
    with torch.no_grad():
        for batch in val_dataloader:
            pixel_values = batch["pixel_values"].to(device)
            labels = batch["labels"].to(device)

            logits = model(pixel_values)
            loss = loss_fn(logits, labels)

            val_loss += loss.item()

    avg_val_loss = val_loss / len(val_dataloader)

    print(f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
    train_losses.append(avg_train_loss)
    val_losses.append(avg_val_loss)

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), os.path.join(drive_path, "best_model.pt"))
        epochs_no_improve = 0
        print("Validation loss improved. Model saved.")
    else:
        epochs_no_improve += 1
        print(f"No improvement for {epochs_no_improve} epoch(s).")

    if epochs_no_improve >= patience:
        print("Early stopping triggered.")
        break


In [ ]:
# Load the best model weights from Drive
print("Loading best model from disk...")
model.load_state_dict(torch.load(os.path.join(drive_path, "best_model.pt")))
model.to(device)
model.eval()


In [ ]:
# Plot Training vs Validation Loss
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.plot(train_losses, label='Training Loss')
plt.plot(val_losses, label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training vs Validation Loss')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
# Evaluation (example)
total_correct = 0
total_pixels = 0

model.eval()
with torch.no_grad():
    for batch in tqdm(test_dataloader):
        pixel_values = batch["pixel_values"].to(device)
        labels = batch["labels"].to(device)

        # Forward pass — model returns logits directly
        logits = model(pixel_values)

        # Resize logits to match original mask size
        upsampled_logits = torch.nn.functional.interpolate(
            logits,
            size=labels.shape[-2:],  # height x width of original mask
            mode="bilinear",
            align_corners=False,
        )

        # Get predicted class per pixel
        predicted_masks = upsampled_logits.argmax(dim=1)

        # Calculate pixel-wise accuracy
        total_correct += (predicted_masks == labels).sum().item()
        total_pixels += labels.numel()

# Final accuracy
accuracy = total_correct / total_pixels
print(f"Pixel-wise Accuracy: {accuracy:.4f}")


In [ ]:
accuracy = total_correct / total_pixels
print(f"Test Accuracy: {accuracy:.4f}")

In [ ]:
def compute_mIoU(preds, labels, num_classes):
    iou_list = []
    preds = preds.view(-1)
    labels = labels.view(-1)
    for cls in range(num_classes):
        pred_inds = preds == cls
        target_inds = labels == cls
        intersection = (pred_inds & target_inds).sum().item()
        union = (pred_inds | target_inds).sum().item()
        if union == 0:
            iou = float('nan')  # ignore classes not present in batch
        else:
            iou = intersection / union
        iou_list.append(iou)
    return iou_list, np.nanmean(iou_list)

def compute_dice(preds, labels, num_classes):
    dice_scores = []
    preds = preds.view(-1)
    labels = labels.view(-1)
    for cls in range(num_classes):
        pred_inds = preds == cls
        target_inds = labels == cls
        intersection = (pred_inds & target_inds).sum().item()
        pred_sum = pred_inds.sum().item()
        target_sum = target_inds.sum().item()
        if pred_sum + target_sum == 0:
            dice = float('nan')
        else:
            dice = 2 * intersection / (pred_sum + target_sum)
        dice_scores.append(dice)
    return dice_scores, np.nanmean(dice_scores)

def per_class_accuracy(preds, labels, num_classes):
    accs = []
    preds = preds.view(-1)
    labels = labels.view(-1)
    for cls in range(num_classes):
        cls_mask = labels == cls
        correct = (preds[cls_mask] == labels[cls_mask]).sum().item()
        total = cls_mask.sum().item()
        acc = correct / total if total != 0 else float('nan')
        accs.append(acc)
    return accs


In [ ]:
class_names = ["Crops", "Bare", "Built", "Water"]


In [ ]:
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in test_dataloader:
        pixel_values = batch["pixel_values"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(pixel_values)  # ← custom MobileViT model returns raw logits
        logits = outputs
        upsampled_logits = torch.nn.functional.interpolate(
            logits, size=labels.shape[-2:], mode="bilinear", align_corners=False)
        predicted_masks = upsampled_logits.argmax(dim=1)

        all_preds.append(predicted_masks.cpu())
        all_labels.append(labels.cpu())

all_preds = torch.cat(all_preds, dim=0)
all_labels = torch.cat(all_labels, dim=0)

iou_per_class, mean_iou = compute_mIoU(all_preds, all_labels, num_labels)
dice_per_class, mean_dice = compute_dice(all_preds, all_labels, num_labels)
accs = per_class_accuracy(all_preds, all_labels, num_labels)

print("Per-Class IoU:")
for i, iou in enumerate(iou_per_class):
    name = class_names[i]
    print(f"  {i} ({name}): {iou:.4f}")

print("\nPer-Class Dice Coefficient:")
for i, dice in enumerate(dice_per_class):
    name = class_names[i]
    print(f"  {i} ({name}): {dice:.4f}")

print("\nPer-Class Accuracy:")
for i, acc in enumerate(accs):
    name = class_names[i]
    print(f"  {i} ({name}): {acc:.4f}")

print(f"\nMean IoU: {mean_iou:.4f}")
print(f"Mean Dice Coefficient: {mean_dice:.4f}")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

class_names = ["Crops", "Bare", "Built", "Water"]
class_colors = [
    (0, 255, 0),      # Crops - Green
    (210, 180, 140),  # Bare - Tan
    (128, 128, 128),  # Built - Gray
    (0, 0, 255),      # Water - Blue
]

def colorize_mask(mask):
    """Convert a class-index mask (H, W) to an RGB mask (H, W, 3) using class_colors."""
    h, w = mask.shape
    color_mask = np.zeros((h, w, 3), dtype=np.uint8)
    for class_idx, color in enumerate(class_colors):
        color_mask[mask == class_idx] = color
    return color_mask


In [ ]:

model.eval()
num_visualizations = 40

with torch.no_grad():
    for i, batch in enumerate(test_dataloader):
        if i >= num_visualizations:
            break

        pixel_values = batch["pixel_values"].to(device)
        labels = batch["labels"].to(device)

        logits = model(pixel_values)

        upsampled_logits = torch.nn.functional.interpolate(
            logits,
            size=labels.shape[-2:],
            mode="bilinear",
            align_corners=False
        )
        predicted = upsampled_logits.argmax(dim=1)

        # Convert tensors to numpy
        image = pixel_values[0].cpu().permute(1, 2, 0).numpy()  # (C, H, W) → (H, W, C)
        label = labels[0].cpu().numpy()
        pred = predicted[0].cpu().numpy()

        # Denormalize image for visualization
        image = (image - image.min()) / (image.max() - image.min())

        # Plot
        fig, axs = plt.subplots(1, 3, figsize=(12, 4))
        axs[0].imshow(image)
        axs[0].set_title("Input Image")
        axs[0].axis('off')

        axs[1].imshow(colorize_mask(label))
        axs[1].set_title("Ground Truth")
        axs[1].axis('off')

        axs[2].imshow(colorize_mask(pred))
        axs[2].set_title("Predicted Mask")
        axs[2].axis('off')

        plt.tight_layout()
        plt.show()


In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

def compute_confusion_matrix(preds, labels, num_classes=4):
    preds_flat = preds.flatten()
    labels_flat = labels.flatten()
    cm = confusion_matrix(labels_flat, preds_flat, labels=range(num_classes))
    return cm

def plot_confusion_matrix(cm, class_names):
    plt.figure(figsize=(8, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names,
                yticklabels=class_names)
    plt.xlabel("Predicted")
    plt.ylabel("Ground Truth")
    plt.title("Confusion Matrix (Pixel-wise)")
    plt.tight_layout()
    plt.show()


In [ ]:
# After collecting all_preds and all_labels (both shape: N, H, W)
all_preds_np = np.stack([p.cpu().numpy() for p in all_preds])
all_labels_np = np.stack([l.cpu().numpy() for l in all_labels])

cm = compute_confusion_matrix(all_preds_np, all_labels_np, num_classes=5)
plot_confusion_matrix(cm, class_names)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_iou_dice_bars(iou_scores, dice_scores, class_names):
    x = np.arange(len(class_names))
    width = 0.35

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(x - width/2, iou_scores, width, label='IoU', color='skyblue')
    ax.bar(x + width/2, dice_scores, width, label='Dice', color='lightgreen')

    ax.set_ylabel('Score')
    ax.set_title('IoU and Dice Coefficient per Class')
    ax.set_xticks(x)
    ax.set_xticklabels(class_names, rotation=45)
    ax.set_ylim(0, 1)
    ax.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
plot_iou_dice_bars(iou_per_class, dice_per_class, class_names)


In [ ]:
# Save model weights
torch.save(model.state_dict(), os.path.join(drive_path, "mobileViT_lulc_model.pt"))
